In [ ]:
from brainspace.mesh.mesh_io import read_surface
from brainspace.utils.parcellation import map_to_labels
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from surfplot import Plot

import cmasher as cmr
import glob
import matplotlib.pyplot as plt
import natsort
import nibabel as nib
import numpy as np
import os
import pandas as pd
import pickle
import seaborn as sns
import SUITPy.flatmap as flatmap

os.makedirs("../figures/resting", exist_ok=True)

def plot_subcortex(data, color_range, save_dir, surf_file='../utils/atlases/subcorticalSurf.pkl'):
    with open(surf_file, 'rb') as f:
        VL, VR = pickle.load(f)

    nROI = len(VL)
    cmap = cmr.get_sub_cmap('twilight_shifted', 0.01, 0.99)
    fig = plt.figure(figsize=(12, 5))
    ax1 = fig.add_subplot(141, projection='3d')
    ax2 = fig.add_subplot(142, projection='3d')
    ax3 = fig.add_subplot(143, projection='3d')
    ax4 = fig.add_subplot(144, projection='3d')

    def val_to_color(val):
        frac = (val - color_range[0]) / float(color_range[1] - color_range[0])
        frac_clamped = max(min(frac, 1.0), 0.0)
        return cmap(frac_clamped)

    coords_ax1 = []
    coords_ax2 = []
    coords_ax3 = []
    coords_ax4 = []

    for p in range(nROI):
        if VL[p]['simplices'].size > 0:
            coords = VL[p]['coords']
            faces = VL[p]['simplices']
            color_val = data[p + nROI]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax1.add_collection3d(mesh)
            coords_ax1.append(coords)  

    ax1.view_init(elev=0, azim=-0)
    ax1.set_box_aspect((1, 1, 1))
    ax1.set_axis_off()

    for p in range(nROI):
        if VL[p]['simplices'].size > 0:
            coords = VL[p]['coords']
            faces = VL[p]['simplices']
            color_val = data[p + nROI]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax2.add_collection3d(mesh)
            coords_ax2.append(coords)

    ax2.view_init(elev=0, azim=180)
    ax2.set_box_aspect((1, 1, 1))
    ax2.set_axis_off()

    for p in range(nROI):
        if VR[p]['simplices'].size > 0:
            coords = VR[p]['coords']
            faces = VR[p]['simplices']
            color_val = data[p]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax3.add_collection3d(mesh)
            coords_ax3.append(coords)

    ax3.view_init(elev=0, azim=0)
    ax3.set_box_aspect((1, 1, 1))
    ax3.set_axis_off()

    for p in range(nROI):
        if VR[p]['simplices'].size > 0:
            coords = VR[p]['coords']
            faces = VR[p]['simplices']
            color_val = data[p]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax4.add_collection3d(mesh)
            coords_ax4.append(coords)

    ax4.view_init(elev=0, azim=180)
    ax4.set_box_aspect((1, 1, 1))
    ax4.set_axis_off()

    if coords_ax1:
        all_coords_1 = np.concatenate(coords_ax1, axis=0)
        ax1.auto_scale_xyz(all_coords_1[:,0], all_coords_1[:,1], all_coords_1[:,2])
    if coords_ax2:
        all_coords_2 = np.concatenate(coords_ax2, axis=0)
        ax2.auto_scale_xyz(all_coords_2[:,0], all_coords_2[:,1], all_coords_2[:,2])
        
    if coords_ax3:
        all_coords_3 = np.concatenate(coords_ax3, axis=0)
        ax3.auto_scale_xyz(all_coords_3[:,0], all_coords_3[:,1], all_coords_3[:,2])
    if coords_ax4:
        all_coords_4 = np.concatenate(coords_ax4, axis=0)
        ax4.auto_scale_xyz(all_coords_4[:,0], all_coords_4[:,1], all_coords_4[:,2])

    plt.tight_layout()
    plt.savefig(f"{save_dir}", dpi=800, transparent=True)
    plt.close()
    
def plot_cerebellum(data_array, color_range, save_dir):
    label_file = '../utils/atlases/atl-NettekovenAsym32_dseg.label.gii'
    labels = nib.load(label_file)
    num_vertices = labels.darrays[0].data.shape[0]
    data = np.zeros(num_vertices)
    roi_values = data_array
    label_data = labels.darrays[0].data
    
    for i, roi_value in enumerate(roi_values):
        data[label_data == (i + 1)] = roi_value  
    cmap = cmr.get_sub_cmap('twilight_shifted', 0.01, 0.99)
    ax = flatmap.plot(data, overlay_type='func', cmap=cmap, cscale=color_range,
                      new_figure=True, colorbar=True, bordercolor='w')
    
    plt.savefig(f"{save_dir}", dpi=800, transparent=True)
    plt.close()

def plot_variance_explained():
    plt.rcParams['font.family'] = 'Times New Roman'
    eigenvals = pd.read_table("../output/pca-gradients-centered/reference_eigenvalues.tsv", index_col=0)['proportion'][:20]
    k = len(eigenvals)
    fig, ax = plt.subplots(figsize=(2, 2.7))
    ax.plot(np.arange(k) + 1, eigenvals, marker='o', markersize=4, markerfacecolor='white', color='black', linewidth=1)
    ax.set(ylabel='% Variance expl.', xlabel='Principal\ncomponent (PC)', xticks=np.arange(0, 21, 5), yticks=np.arange(0, .28, .05))
    sns.despine()
    fig.tight_layout(w_pad=3)
    plt.savefig("../figures/resting/variance_explained.png", dpi=800)
    plt.close()

def load_gradients(fname, k=None):
    df = pd.read_table(fname, index_col=0)
    if k is not None:
        df = df.iloc[:, :k]
    df = df.reset_index().rename(columns={'index': 'roi'})
    return df

def plot_cbar(cmap, vmin, vmax, orientation='vertical', size=None, n_ticks=2, 
              decimals=2, fontsize=12, show_outline=False, as_int=False):
    if size is None and (orientation == 'vertical'):
        size = (.3, 4)
    if size is None and (orientation == 'horizontal'):
        size = (4, .3)
    x = np.array([[0,1]])
    plt.figure(figsize=size)
    img = plt.imshow(x, cmap=cmap)
    plt.gca().set_visible(False)
    cax = plt.axes([0.1, 0.2, 0.8, 0.6])

    ticks = np.linspace(0, 1, n_ticks)
    tick_labels = np.around(np.linspace(vmin, vmax, n_ticks), decimals)
    cbar = plt.colorbar(orientation=orientation, cax=cax, ticks=ticks)
    if as_int:
        cbar.set_ticklabels(tick_labels.astype(int))
    else:
        cbar.set_ticklabels(tick_labels)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    
    if not show_outline:
        cbar.outline.set_visible(False)
        cbar.ax.tick_params(size=0)
    return cbar

def get_surfaces(style='inflated', load=True):
    surf_path = '../utils/surfaces'
    surfaces = get_files([surf_path, f'*.{style}_*'])
    if load:
        surfs = [read_surface(i) for i in surfaces]
        return dict(zip(['lh', 'rh'], surfs))
    else:
        return surfaces

def get_files(pattern, force_list=False):
    if isinstance(pattern, list):
        pattern = os.path.join(*pattern)
    files = natsort.natsorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError('Pattern could not detect file(s)')

    if (not force_list) & (len(files) == 1):
        return files[0] 
    else:
        return files

def weights_to_vertices(data, target, labels=None):
    if isinstance(target, str): 
        vertices = nib.load(target).get_fdata().ravel()
    else:
        vertices = target.ravel()
    if labels is not None:
        data = _align_labels_to_atlas(data, labels, vertices)
    mask = vertices != 0
    map_args = dict(target_lab=vertices, mask=mask, fill=np.nan)
    if (len(data.shape) == 1) or (data.shape[1] == 1):
        weights = map_to_labels(data.ravel(),  **map_args)
    else:
        weights = [map_to_labels(x, **map_args) for x in data.T]
    return weights
    
def plot_resting_brain_gradients():
    gradients = load_gradients("../output/pca-gradients-centered/reference_gradient.tsv", 5)
    
    cerebellum_gradients = gradients.iloc[:32]
    subcortex_gradients = gradients.iloc[32:64]
    cortical_gradients = gradients.iloc[64:]
    cmap = cmr.get_sub_cmap('twilight_shifted', 0.01, 0.99)

    cortical_grads = cortical_gradients.filter(like='g').values
    cerebellum_grads = cerebellum_gradients.filter(like='g').values
    subcortex_grads = subcortex_gradients.filter(like='g').values
    grads = gradients.filter(like='g').values

    vmax = np.max(grads)
    plot_cbar(cmap, -vmax, vmax, 'horizontal', size=(1, .3), n_ticks=3)
    plt.savefig("../figures/resting/resting_cbar.png", dpi=800, transparent=True)

    surfaces = get_surfaces()
    for i in range(5):
        x = weights_to_vertices(cortical_grads[:, i], "../utils/atlases/Schaefer2018_400Parcels_7Networks_order.dlabel.nii")
        p = Plot(surfaces['lh'], surfaces['rh'])
        p.add_layer(x, cmap=cmap, color_range=(-vmax, vmax), cbar=False)
        fig = p.build(colorbar=False)
        fig.savefig(f'../figures/resting/PC{i + 1}_brain.png', dpi=800, transparent=True)
        plt.close(fig)
        plot_cerebellum(cerebellum_grads[:, i], [-vmax, vmax], f"../figures/resting/PC{i + 1}_cerebellum.png") 
        plot_subcortex(subcortex_grads[:, i], [-vmax, vmax], f"../figures/resting/PC{i + 1}_subcortex.png")

def extract_network(roi):
    if roi.startswith('7Networks_'):
        return roi.split('_')[2] 
    elif '*' in roi:
        return roi.split('*')[1] 
    else:
        return roi  

def yeo_cmap():
    cmap = {'cerebellum': (0, 0, 0), 'subcortex': (77, 77, 77), 'Vis': (119, 20, 140), 'SomMot': (70, 126, 175), 
           'DorsAttn': (0, 117, 7), 'SalVentAttn': (195, 59, 255), 'Limbic': (219, 249, 165), 'Cont': (230, 149, 33), 
           'Default': (205, 65, 80)}
    cmap = {k: np.array(v) / 255 for k, v in cmap.items()}
    return cmap

def plot_3d(x, y, z, color=None, ax=None, view_3d=(35, -110), **kwargs):
    if ax is None:
        fig = plt.figure(figsize=(4, 4), frameon=False)
        ax = fig.add_subplot(projection='3d')
    
    ax.scatter(xs=x, ys=y, zs=z, c=color, **kwargs)
    ax.set(xlabel='PC1', ylabel='PC2', zlabel='PC3')
    if view_3d is not None:
        ax.view_init(elev=view_3d[0], azim=view_3d[1])
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.grid(False)
    return ax

def plot_reference_manifold():
    df = load_gradients("../output/pca-gradients-centered/reference_gradient.tsv", 5)
    schaefer_rois = pd.read_csv("../utils/atlases/schaefer-parcellation.csv")["name_7networks"].to_list()
    cerebellum_rois = ["cerebellum"] * 32
    subcortical_rois = ["subcortex"] * 32
    roi_list = cerebellum_rois + subcortical_rois + schaefer_rois
    df["roi"] = roi_list

    df['network'] = df['roi'].apply(extract_network)
    cmap = yeo_cmap()
    df['c'] = df['network'].apply(lambda x: cmap[x])

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(df['g1'], df['g2'], df['g3'], c=df['c'], s=20, edgecolor='none')
    ax.set_xlabel("PC1", labelpad=10)
    ax.set_ylabel("PC2", labelpad=10)
    ax.set_zlabel("PC3", labelpad=10)
    ax.grid(False)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_zlim(-3, 3)
    ax.view_init(elev=20, azim=-150)
    plt.tight_layout()
    plt.savefig("../figures/resting/resting_manifold_3d.png", dpi=800, transparent=True)
    plt.close()

    df = load_gradients("../output/pca-gradients-centered/reference_gradient.tsv", 5)
    schaefer_rois = pd.read_csv("../utils/atlases/schaefer-parcellation.csv")["name_7networks"].to_list()
    cerebellum_rois = ["cerebellum"] * 32
    subcortical_rois = ["subcortex"] * 32
    roi_list = cerebellum_rois + subcortical_rois + schaefer_rois
    df["roi"] = roi_list

    df['network'] = df['roi'].apply(extract_network)
    cmap = yeo_cmap()
    df['c'] = df['network'].apply(lambda x: cmap[x])

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(df['g5'], df['g4'], df['g3'], c=df['c'], s=20, edgecolor='none')
    ax.set_xlabel("PC5", labelpad=10)
    ax.set_ylabel("PC4", labelpad=10)
    ax.set_zlabel("PC3", labelpad=10)
    ax.grid(False)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_zlim(-3, 3)
    ax.view_init(elev=20, azim=-150)
    plt.tight_layout()
    plt.savefig("../figures/resting/resting_manifold_3d_g5.png", dpi=800, transparent=True)
    plt.close()
    
    df = load_gradients("../output/pca-gradients-centered/reference_gradient.tsv", 5)
    schaefer_rois = pd.read_csv("../utils/atlases/schaefer-parcellation.csv")["name_7networks"].to_list()
    df["roi"] = ["cerebellum"] * 32 + ["subcortex"] * 32 + schaefer_rois
    df['network'] = df['roi'].apply(extract_network)
    cmap = yeo_cmap()

    plt.rcParams['font.family'] = 'Times New Roman'
    os.makedirs("../figures/resting", exist_ok=True)

    for col, label in [('g1', 'PC1'), ('g2', 'PC2'), ('g3', 'PC3'), ('g4', 'PC4'), ('g5', 'PC5')]:
        fig, ax = plt.subplots(figsize=(5, 2))
        sns.kdeplot(data=df, x=col, hue='network', palette=cmap, fill=True, common_norm=False,
                    alpha=0.6, linewidth=1, ax=ax, legend=False)
        sns.despine(ax=ax)
        fig.tight_layout()
        sns.despine(ax=ax, left=True, bottom=True)
        ax.set_xlim(-4, 4)
       # ax.set(xlabel=None, ylabel=None, xticks=[], yticks=[])
        plt.savefig(f"../figures/resting/{label}_kde.png", dpi=800, transparent=True)
        plt.close(fig)

    df = load_gradients("../output/pca-gradients-centered/reference_gradient.tsv", 5)
    schaefer_rois   = pd.read_csv("../utils/atlases/schaefer-parcellation.csv")["name_7networks"].to_list()
    cerebellum_rois = ["cerebellum"] * 32
    subcortical_rois= ["subcortex"] * 32
    df["roi"]       = cerebellum_rois + subcortical_rois + schaefer_rois
    df["network"]   = df["roi"].apply(extract_network)
    cmap            = yeo_cmap()
    df["c"]         = df["network"].apply(lambda x: cmap[x])

    plt.rcParams['font.family'] = 'Times New Roman'
    pairs = [("g1", "g2", "PC1", "PC2"), ("g1", "g3", "PC1", "PC3"), ("g1", "g4", "PC1", "PC4"), ("g1", "g5", "PC1", "PC5"),
             ("g2", "g3", "PC2", "PC3"), ("g2", "g4", "PC2", "PC4"), ("g2", "g5", "PC2", "PC5"), ("g3", "g4", "PC3", "PC4"),
             ("g3", "g5", "PC3", "PC5"), ("g4", "g5", "PC4", "PC5")]
    
    n_plots = len(pairs)
    n_cols = 5
    n_rows = int(np.ceil(n_plots / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows), sharex=True, sharey=True, constrained_layout=True)
    axes = axes.flatten()

    for ax, (xcol, ycol, xl, yl) in zip(axes, pairs):
        ax.scatter(df[xcol], df[ycol], c=df["c"], s=12, edgecolor='none')
        ax.set_xlabel(xl)
        ax.set_ylabel(yl)
        ax.grid(False)
        ax.set_aspect('equal', adjustable='box')
        ax.margins(0)
        ax.use_sticky_edges = False
    
    for ax in axes[len(pairs):]:
        ax.axis('off')
    
    for ax in axes[:len(pairs)]:
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.autoscale(False)
    
    plt.tight_layout()
    out = "../figures/resting/resting_manifold_pairs.png"
    plt.savefig(out, dpi=800, transparent=True)
    plt.close(fig)

def main():
    os.makedirs("../figures/resting", exist_ok=True)
    plot_variance_explained()
    plot_resting_brain_gradients()
    plot_reference_manifold()
main()